<h1><center><b><font color=#3FA1BF>Álgebra Lineal — Espacios Euclídeos</font></b></center></h1>
<h2><center><font color=#3FA1BF>Actividad 3</font></center></h2>

Esta libreta resuelve los ejercicios de la **Actividad 3** del curso *Matemáticas para IA* (UAM), centrados en el estudio de **espacios euclídeos**.

Conceptos clave:
- **Norma euclídea:** $\|x\| = \sqrt{\langle x,\, x \rangle}$
- **Producto interno estándar:** $\langle u, v \rangle = \displaystyle\sum_{i} u_i v_i$
- **Ortogonalidad:** $u \perp v \iff \langle u, v \rangle = 0$
- **Proceso de Gram-Schmidt:** construye una base ortogonal a partir de cualquier conjunto generador

In [ ]:
import numpy as np
from numpy import linalg
import sympy

sympy.init_printing(use_latex='mathjax')

---
## <center><font color=#3FA1BF>Ejercicio 1 — Normas y Productos Internos</font></center>

Dados los vectores:
$$x_1 = (1,\, 3,\, -2), \quad x_2 = (-3,\, 4,\, 5), \quad x_3 = \left(\tfrac{1}{2},\, 2,\, -3\right)$$

Determinar:

**(a)** Las **normas euclídeas** $\|x_1\|$, $\|x_2\|$ y $\|x_3\|$, donde:
$$\|x\| = \sqrt{x_1^2 + x_2^2 + \cdots + x_n^2}$$

**(b)** Los **productos internos** $\langle x_1, x_2 \rangle$, $\langle x_1, x_3 \rangle$ y $\langle x_2, x_2 \rangle$, donde:
$$\langle u, v \rangle = \sum_{i=1}^n u_i\, v_i$$

In [ ]:
# Ejercicio 1 — Normas euclídeas y productos internos
import numpy as np
import sympy

# Vectores en NumPy (cálculo numérico)
x1_np = np.array([1, 3, -2], dtype=float)
x2_np = np.array([-3, 4, 5], dtype=float)
x3_np = np.array([0.5, 2, -3], dtype=float)

# Vectores en SymPy (cálculo simbólico exacto — x3 usa Rational para la componente 1/2)
x1_sp = sympy.Matrix([1, 3, -2])
x2_sp = sympy.Matrix([-3, 4, 5])
x3_sp = sympy.Matrix([sympy.Rational(1, 2), 2, -3])

# --- (a) Normas euclídeas ---
print('=== (a) Normas euclídeas ===')
for nombre, v_np, v_sp in [('x1', x1_np, x1_sp), ('x2', x2_np, x2_sp), ('x3', x3_np, x3_sp)]:
    norma_np = np.linalg.norm(v_np)
    norma_sp = sympy.simplify(v_sp.norm())
    print(f'||{nombre}|| (NumPy) = {norma_np:.6f}   ||{nombre}|| (SymPy exacto) = {norma_sp}')

# --- (b) Productos internos ---
print()
print('=== (b) Productos internos ===')
pares = [('x1', 'x2', x1_np, x2_np, x1_sp, x2_sp),
         ('x1', 'x3', x1_np, x3_np, x1_sp, x3_sp),
         ('x2', 'x2', x2_np, x2_np, x2_sp, x2_sp)]
for n1, n2, v1n, v2n, v1s, v2s in pares:
    prod_np = np.dot(v1n, v2n)
    prod_sp = v1s.dot(v2s)
    print(f'<{n1}, {n2}> (NumPy) = {prod_np}   <{n1}, {n2}> (SymPy exacto) = {prod_sp}')

print()
print(f'Nota: <x2, x2> = ||x2||^2 = {x2_sp.dot(x2_sp)} (cuadrado de la norma) ✓')

=== (a) Normas euclídeas ===
||x1|| (NumPy) = 3.741657   ||x1|| (SymPy exacto) = sqrt(14)
||x2|| (NumPy) = 7.071068   ||x2|| (SymPy exacto) = 5*sqrt(2)
||x3|| (NumPy) = 3.640055   ||x3|| (SymPy exacto) = sqrt(53)/2

=== (b) Productos internos ===
<x1, x2> (NumPy) = -1.0   <x1, x2> (SymPy exacto) = -1
<x1, x3> (NumPy) = 12.5   <x1, x3> (SymPy exacto) = 25/2
<x2, x2> (NumPy) = 50.0   <x2, x2> (SymPy exacto) = 50

Nota: <x2, x2> = ||x2||^2 = 50 (cuadrado de la norma) ✓


---
## <center><font color=#3FA1BF>Ejercicio 2 — Base Ortogonal (Proceso de Gram-Schmidt)</font></center>

En cada caso, encuentre una **base ortogonal** del subespacio generado por los vectores dados.

### Proceso de Gram-Schmidt

Dado un conjunto $\{x_1, x_2, \ldots, x_n\}$, se construye una base ortogonal $\{u_1, u_2, \ldots, u_k\}$ con $k \leq n$:

$$u_1 = x_1$$

$$u_j = x_j - \sum_{i=1}^{j-1} \frac{\langle x_j,\, u_i \rangle}{\langle u_i,\, u_i \rangle}\, u_i \qquad (j = 2, 3, \ldots)$$

Si $u_j = \mathbf{0}$, el vector $x_j$ es **linealmente dependiente** de los anteriores y se descarta.

---

**(a)** $x_1 = (1,1,1),\quad x_2 = (1,0,1),\quad x_3 = (3,2,3)$

**(b)** $x_1 = (1,1,1),\quad x_2 = (-1,1,-1),\quad x_3 = (1,0,1)$

**(c)** $x_1 = (1,1,0,0),\quad x_2 = (0,1,1,0),\quad x_3 = (0,0,1,1),\quad x_4 = (1,0,0,1)$

In [ ]:
# Funciones auxiliares para Gram-Schmidt y verificación de ortogonalidad
import numpy as np
import sympy
from sympy.matrices.dense import MutableDenseMatrix

def gram_schmidt_manual(vecs):
    # Proceso de Gram-Schmidt (versión clásica) sobre arrays NumPy.
    # Fórmula: u_j = x_j - sum_i ( <x_j, u_i> / <u_i, u_i> ) * u_i
    # Descarta vectores cuya norma cae por debajo de 1e-10 (dependencia lineal).
    ortho = []
    for i, v in enumerate(vecs):
        w = v.astype(float).copy()
        for u in ortho:
            w = w - (np.dot(v, u) / np.dot(u, u)) * u
        if np.linalg.norm(w) > 1e-10:
            ortho.append(w)
        else:
            print(f'  Info: x{i+1} es linealmente dependiente (vector nulo — se descarta)')
    return ortho

def verificar_ortogonalidad(base):
    # Comprueba que ui · uj = 0 para todo par i ≠ j.
    print('Verificación de ortogonalidad:')
    ok = True
    for i in range(len(base)):
        for j in range(i + 1, len(base)):
            prod = round(np.dot(base[i], base[j]), 12)
            estado = '[OK]' if abs(prod) < 1e-10 else '[FALLA]'
            print(f'  u{i+1} · u{j+1} = {prod:.2e}  {estado}')
            if abs(prod) >= 1e-10:
                ok = False
    res = 'Si — base ortogonal' if ok else 'No'
    print(f'  Resultado: {res}')

def gram_schmidt_sympy(vecs_sp):
    # GramSchmidt simbólico con rankcheck=False para aceptar vectores
    # linealmente dependientes (los vectores nulos se descartan automáticamente).
    return MutableDenseMatrix.orthogonalize(*vecs_sp, normalize=False, rankcheck=False)

print('Funciones gram_schmidt_manual, verificar_ortogonalidad y gram_schmidt_sympy cargadas.')

Funciones gram_schmidt_manual, verificar_ortogonalidad y gram_schmidt_sympy cargadas.


In [ ]:
# Ejercicio 2a — Base ortogonal de {(1,1,1), (1,0,1), (3,2,3)}
print('=' * 58)
print('Ejercicio 2a: x1=(1,1,1), x2=(1,0,1), x3=(3,2,3)')
print('=' * 58)

vecs = [np.array([1, 1, 1]),
        np.array([1, 0, 1]),
        np.array([3, 2, 3])]

print()
print('--- Gram-Schmidt (NumPy) ---')
base = gram_schmidt_manual(vecs)
print(f'Base ortogonal del subespacio ({len(base)} vectores):')
for i, u in enumerate(base):
    print(f'  u{i+1} = {u}')
verificar_ortogonalidad(base)

print()
print('--- Verificación simbólica exacta (SymPy) ---')
vecs_sp = [sympy.Matrix(v) for v in vecs]
base_sp = gram_schmidt_sympy(vecs_sp)
print(f'Base ortogonal simbólica ({len(base_sp)} vectores):')
for i, u in enumerate(base_sp):
    print(f'  u{i+1} = {u.T}')
print('Productos internos cruzados (deben ser 0):')
for i in range(len(base_sp)):
    for j in range(i + 1, len(base_sp)):
        prod = base_sp[i].dot(base_sp[j])
        print(f'  u{i+1}·u{j+1} = {prod}')

Ejercicio 2a: x1=(1,1,1), x2=(1,0,1), x3=(3,2,3)

--- Gram-Schmidt (NumPy) ---
  Info: x3 es linealmente dependiente (vector nulo — se descarta)
Base ortogonal del subespacio (2 vectores):
  u1 = [1. 1. 1.]
  u2 = [ 0.33333333 -0.66666667  0.33333333]
Verificación de ortogonalidad:
  u1 · u2 = 0.00e+00  [OK]
  Resultado: Si — base ortogonal

--- Verificación simbólica exacta (SymPy) ---
Base ortogonal simbólica (2 vectores):
  u1 = Matrix([[1, 1, 1]])
  u2 = Matrix([[1/3, -2/3, 1/3]])
Productos internos cruzados (deben ser 0):
  u1·u2 = 0


In [ ]:
# Ejercicio 2b — Base ortogonal de {(1,1,1), (-1,1,-1), (1,0,1)}
print('=' * 58)
print('Ejercicio 2b: x1=(1,1,1), x2=(-1,1,-1), x3=(1,0,1)')
print('=' * 58)

vecs = [np.array([ 1, 1,  1]),
        np.array([-1, 1, -1]),
        np.array([ 1, 0,  1])]

print()
print('--- Gram-Schmidt (NumPy) ---')
base = gram_schmidt_manual(vecs)
print(f'Base ortogonal del subespacio ({len(base)} vectores):')
for i, u in enumerate(base):
    print(f'  u{i+1} = {u}')
verificar_ortogonalidad(base)

print()
print('--- Verificación simbólica exacta (SymPy) ---')
vecs_sp = [sympy.Matrix(v) for v in vecs]
base_sp = gram_schmidt_sympy(vecs_sp)
print(f'Base ortogonal simbólica ({len(base_sp)} vectores):')
for i, u in enumerate(base_sp):
    print(f'  u{i+1} = {u.T}')
print('Productos internos cruzados (deben ser 0):')
for i in range(len(base_sp)):
    for j in range(i + 1, len(base_sp)):
        prod = base_sp[i].dot(base_sp[j])
        print(f'  u{i+1}·u{j+1} = {prod}')

Ejercicio 2b: x1=(1,1,1), x2=(-1,1,-1), x3=(1,0,1)

--- Gram-Schmidt (NumPy) ---
  Info: x3 es linealmente dependiente (vector nulo — se descarta)
Base ortogonal del subespacio (2 vectores):
  u1 = [1. 1. 1.]
  u2 = [-0.66666667  1.33333333 -0.66666667]
Verificación de ortogonalidad:
  u1 · u2 = -0.00e+00  [OK]
  Resultado: Si — base ortogonal

--- Verificación simbólica exacta (SymPy) ---
Base ortogonal simbólica (2 vectores):
  u1 = Matrix([[1, 1, 1]])
  u2 = Matrix([[-2/3, 4/3, -2/3]])
Productos internos cruzados (deben ser 0):
  u1·u2 = 0


In [ ]:
# Ejercicio 2c — Base ortogonal de 4 vectores en R⁴
print('=' * 64)
print('Ejercicio 2c: 4 vectores en R^4')
print('  x1=(1,1,0,0), x2=(0,1,1,0), x3=(0,0,1,1), x4=(1,0,0,1)')
print('=' * 64)

vecs = [np.array([1, 1, 0, 0]),
        np.array([0, 1, 1, 0]),
        np.array([0, 0, 1, 1]),
        np.array([1, 0, 0, 1])]

print()
print('--- Gram-Schmidt (NumPy) ---')
base = gram_schmidt_manual(vecs)
print(f'Base ortogonal del subespacio ({len(base)} vectores):')
for i, u in enumerate(base):
    print(f'  u{i+1} = {u}')
verificar_ortogonalidad(base)

print()
print('--- Verificación simbólica exacta (SymPy) ---')
vecs_sp = [sympy.Matrix(v) for v in vecs]
base_sp = gram_schmidt_sympy(vecs_sp)
print(f'Base ortogonal simbólica ({len(base_sp)} vectores):')
for i, u in enumerate(base_sp):
    print(f'  u{i+1} = {u.T}')
print('Productos internos cruzados (deben ser 0):')
for i in range(len(base_sp)):
    for j in range(i + 1, len(base_sp)):
        prod = base_sp[i].dot(base_sp[j])
        print(f'  u{i+1}·u{j+1} = {prod}')

Ejercicio 2c: 4 vectores en R^4
  x1=(1,1,0,0), x2=(0,1,1,0), x3=(0,0,1,1), x4=(1,0,0,1)

--- Gram-Schmidt (NumPy) ---
  Info: x4 es linealmente dependiente (vector nulo — se descarta)
Base ortogonal del subespacio (3 vectores):
  u1 = [1. 1. 0. 0.]
  u2 = [-0.5  0.5  1.   0. ]
  u3 = [ 0.33333333 -0.33333333  0.33333333  1.        ]
Verificación de ortogonalidad:
  u1 · u2 = 0.00e+00  [OK]
  u1 · u3 = 0.00e+00  [OK]
  u2 · u3 = 0.00e+00  [OK]
  Resultado: Si — base ortogonal

--- Verificación simbólica exacta (SymPy) ---
Base ortogonal simbólica (3 vectores):
  u1 = Matrix([[1, 1, 0, 0]])
  u2 = Matrix([[-1/2, 1/2, 1, 0]])
  u3 = Matrix([[1/3, -1/3, 1/3, 1]])
Productos internos cruzados (deben ser 0):
  u1·u2 = 0
  u1·u3 = 0
  u2·u3 = 0
